In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv('../Datasets/Iris.csv')
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             150 non-null    int64  
 1   SepalLengthCm  150 non-null    float64
 2   SepalWidthCm   150 non-null    float64
 3   PetalLengthCm  150 non-null    float64
 4   PetalWidthCm   150 non-null    float64
 5   Species        150 non-null    object 
dtypes: float64(4), int64(1), object(1)
memory usage: 7.2+ KB


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,75.500000,5.843333,3.054000,3.758667,1.198667
std,43.445368,0.828066,0.433594,1.764420,0.763161
min,1.000000,4.300000,2.000000,1.000000,0.100000
25%,38.250000,5.100000,2.800000,1.600000,0.300000
50%,75.500000,5.800000,3.000000,4.350000,1.300000
75%,112.750000,6.400000,3.300000,5.100000,1.800000
max,150.000000,7.900000,4.400000,6.900000,2.500000


In [36]:
df = df.drop(columns=['Id'])

In [37]:
df['Species'] = df['Species'].map({
    'Iris-setosa': 0,
    'Iris-versicolor': 1,
    'Iris-virginica': 2
})

In [38]:
X = df.drop(columns=['Species']).values
y = df['Species'].values

In [39]:
print(X.shape)
print(y.shape)
print(X[:3])
print(y[:3])

(150, 4)
(150,)
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]]
[0 0 0]


In [40]:
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

In [41]:
def get_distances(X_train, x):
    distances = []
    
    for x_train in X_train:
        dist = euclidean_distance(x_train, x)
        distances.append(dist)
    
    return np.array(distances)

In [42]:
def get_k_nearest(distances, k):
    return np.argsort(distances)[:k]

In [ ]:
distances = get_distances(X_train, X)
k_indices = get_k_nearest(distances, k=3)
print(k_indices)

[ 0 17  4]


In [44]:
def majority_vote(y_train, k_indices):
    k_labels = y_train[k_indices]
    return np.bincount(k_labels).argmax()

In [45]:
def predict_single(X_train, y_train, x, k):
    distances = get_distances(X_train, x)
    k_indices = get_k_nearest(distances, k)
    label = majority_vote(y_train, k_indices)
    return label

In [46]:
pred = predict_single(X, y, X[0], k=3)
print(pred)

0


In [47]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [48]:
def predict(X_train, y_train, X_test, k):
    predictions = []
    
    for x in X_test:
        label = predict_single(X_train, y_train, x, k)
        predictions.append(label)
    
    return np.array(predictions)

In [49]:
y_pred = predict(X_train, y_train, X_test, k=3)
print(y_pred[:10])

[1 0 2 1 1 0 1 2 1 1]


In [50]:
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

In [55]:
y_pred = predict(X_train, y_train, X_test, k=7)

acc = accuracy(y_test, y_pred)
print(acc)

0.9666666666666667


In [52]:
def confusion_matrix(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    
    for i in range(len(y_true)):
        actual = y_true[i]
        predicted = y_pred[i]
        
        cm[actual][predicted] += 1
    
    return cm

In [53]:
cm = confusion_matrix(y_test, y_pred, num_classes=3)
print(cm)

[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


In [58]:
for k in [1, 3, 5, 7, 9]:
    y_pred = predict(X_train, y_train, X_test, k)
    acc = accuracy(y_test, y_pred)
    print(k, acc)

1 1.0
3 1.0
5 1.0
7 0.9666666666666667
9 1.0
